# TabPFN-3 Regressor — DIMER artifact inference tutorial (standalone)

[![GitHub](https://img.shields.io/badge/GitHub-181717?style=flat&logo=github&logoColor=white)](https://github.com/kurtvalcorza/tabpfn-regressor-pipeline) [![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/kurtvalcorza/tabpfn-regressor-pipeline/blob/main/tutorials/tabpfn_regressor_artifact_inference_colab.ipynb) [![Hugging Face](https://img.shields.io/badge/%F0%9F%A4%97%20Hugging%20Face-Prior--Labs%2Ftabpfn__3-ffcc4d?style=flat)](https://huggingface.co/Prior-Labs/tabpfn_3) [![Upstream](https://img.shields.io/badge/Upstream-PriorLabs%2FTabPFN-181717?style=flat&logo=github&logoColor=white)](https://github.com/PriorLabs/TabPFN) [![arXiv](https://img.shields.io/badge/arXiv-2605.13986-b31b1b.svg)](https://arxiv.org/abs/2605.13986)

**Profile:** `ARTIFACT-INFERENCE`  
**Mode:** `GUIDED`  
**Notebook specification:** DIMER Notebook Specification 2.0 — **standalone** (§4)  
**Capability:** serving-state reconstruction from an externally produced DIMER TabPFN regressor bundle (`artifact_manifest.json` + `model.tabpfn_fit` + `model.ckpt`) and point-estimate inference on genuinely new rows (no prediction intervals)

**This notebook is standalone.** It carries the repository's pipeline module (`src/tabpfn_regressor_pipeline/pipeline.py` at revision `b0f5ba58f9f8`) verbatim in Section 2, the pinned model identity and the per-file SHA-256 manifest in Section 3, and the exact runtime pins in Section 1, so it keeps working after export even if the repository changes or disappears. Its only external dependencies are the pinned Python distributions and the Hugging Face Hub at the immutable revision `24a16a89d245878b846555110985634aa2e656d7` (~233 MB, digest-verified before loading). It was generated by `tools/build_notebook.py` (build_notebook.py/2); edit the repository and regenerate rather than editing cells.

**Run all:** **Known NOTEBOOK_SPEC 2.0 gap (§19, SART1/RUN5/RUN2):** the default path does not yet obtain a trusted sample bundle or sample input automatically — with `ARTIFACT_ZIP_PATH` and `NEW_DATA_PATH` empty, Sections 4 and 6 open upload dialogs for a predictor bundle produced by the E2E tutorial and for unlabelled rows; an executor sets both paths to files already in the runtime to skip the dialogs. Until a published sample bundle and sample rows are wired in, this notebook is a `Candidate`, not release-grade. Once they are present, **Run all** installs the pinned dependencies, validates the bundle (path-safe extraction, manifest digests, provenance, pinned model identity) before any deserialisation, reconstructs the serving estimator from the bundle alone (fitted state and checkpoint digests verified against `EXPECTED_FITTED_SHA256`/`EXPECTED_CKPT_SHA256` when supplied; nothing refit), validates the new rows into an input manifest, emits point predictions (no per-prediction uncertainty), reports what cannot be measured, and exports outputs — all inside this kernel, with no DIMER worker or service and no credential.

**Bring Your Own Data:** New-input BYOD is the `NEW_DATA_PATH`/upload branch in Section 6: your own unlabelled CSV with the bundle's required feature columns passes through the same validation, prediction and export cells. A user-supplied bundle is the separate `ARTIFACT_ZIP_PATH`/upload branch in Section 4 (`EXPECTED_ZIP_SHA256`, `EXPECTED_FITTED_SHA256` and `EXPECTED_CKPT_SHA256` pin it), validated before any state is reconstructed. Uploads stay inside this runtime; do not upload confidential or restricted data unless you are authorised to process it here.

This notebook consumes a DIMER artifact bundle produced **outside this execution** — by the task-inference tutorial in a separate session, or by the DIMER worker, which writes the same three files: `artifact_manifest.json`, `model.tabpfn_fit` (fitted estimator state including the in-context training rows) and `model.ckpt` (foundation weights). It validates the bundle before any model state is deserialised (manifest schema and task type, member names, sizes, SHA-256 digests, archive safety of the fitted ZIP), checks the bundled checkpoint against the pinned TabPFN-3 checkpoint carried by this notebook, reconstructs the estimator through the carried module (`from_artifact`: the fitted archive's recorded `model_path` is rewritten in a temporary copy to the companion checkpoint; TabPFN's `load_fitted_tabpfn_model` restores the state), accepts genuinely new unlabelled rows, predicts point estimates, and exports results. **No training, fine-tuning or in-context refitting occurs, and no artifact is created here.**

**Trust boundary.** Digest checks establish that the three files are internally consistent, not that the sender is trustworthy. `model.tabpfn_fit` is a ZIP of JSON parameters plus serialised Python/torch estimator state and `model.ckpt` is a torch checkpoint; loading them executes trusted model state, and the archive path-safety checks do not change that. The pinned checkpoint of Section 3 is acquired and digest-verified independently so the bundled `model.ckpt` can be required to equal it byte for byte. Load only bundles from a producer you trust, and paste the digests you were given out-of-band into the expected-digest fields. The TabPFN-3 weights are non-commercial (`tabpfn-3-license-v1.0`).

**Learning objectives:** install the pinned runtime, read what the carried module guarantees, resolve and digest-verify the immutable TabPFN-3 checkpoint, supply an externally produced bundle and validate it before any model state is reconstructed, require the bundled checkpoint to equal the pinned one, reconstruct the serving estimator from the bundle alone, validate new unlabelled rows into an input manifest, predict point estimates in target units (no intervals are shipped), produce an evaluation report that is `not-measurable` because no labels exist, and export machine-readable predictions plus provenance.

**This notebook does not demonstrate:** artifact creation, in-notebook support fitting, fine-tuning, classification, prediction intervals or calibrated uncertainty, or any quality claim: without labelled rows nothing is measured, and the exported values are point estimates only.

## Prerequisites

- **Runtime:** a fresh supported runtime (Google Colab or Jupyter, Python 3.11+). The default path runs on CPU and uses CUDA automatically when available.
- **Artifact:** an externally produced bundle ZIP holding `artifact_manifest.json`, `model.tabpfn_fit` and `model.ckpt` (the task-inference tutorial writes `outputs/tabpfn_regressor_artifact.zip`; a DIMER worker's `artifacts/` directory zipped flat works too). Supply it through the upload dialog, or set `ARTIFACT_ZIP_PATH` to a file already present in the runtime for non-interactive execution. Nothing in this notebook manufactures it.
- **Data:** one separate, unlabelled CSV with exactly the bundle's feature columns. It is supplied by upload or by `NEW_DATA_PATH`; no sample is bundled, because scoring self-generated rows would not be external-artifact evidence. Do not upload confidential or restricted data to a hosted notebook environment unless you are authorized to do so. Uploaded inputs remain in the notebook runtime; this pipeline does not send them to a third-party inference API.
- **External access:** the Hugging Face Hub only, to fetch the pinned `Prior-Labs/tabpfn_3` snapshot (~233 MB in total) at revision `24a16a89d245…`. No GitHub access and no credentials are required; nothing is installed from this repository.

## 1. Install the pinned runtime

The dependency set is pinned exactly (the same pins as the repository's pyproject.toml at the generating revision; any `--index-url`/`--find-links` lines are passed to pip as written) and installed directly — there is no repository clone and no package install. If a pin replaces a distribution this runtime has already imported, the cell stops with a restart instruction rather than continuing with mixed versions. Look for a dictionary reporting the notebook's source revision, Python, `torch`, `pandas`, `sklearn` versions, and whether CUDA is available.

In [ ]:
import importlib
import importlib.metadata
import os
import platform
import subprocess
import sys

PINS = [
    'torch==2.11.0',
    'tabpfn==8.1.0',
    'pandas==2.3.2',
    'scikit-learn==1.9.0',
    'huggingface-hub==0.36.2',
]
NOTEBOOK_SOURCE = {
    'repository': 'tabpfn-regressor-pipeline',
    'repository_revision': 'b0f5ba58f9f8e10a21ad276da553c6bd22f014c6',
    'embedded_module': 'src/tabpfn_regressor_pipeline/pipeline.py',
    'embedded_modules': ['src/tabpfn_regressor_pipeline/pipeline.py'],
    'module_sha256': 'e5499d350dc412a97abf431f50ad48fdea4c3dcad042baf487e47ee849f3bea7',
    'generator': 'build_notebook.py/2',
    'notebook_spec': '2.0',
}
SKIP_INSTALL = os.environ.get('DIMER_NOTEBOOK_CI_PREINSTALLED') == '1'

def _installed_version(distribution):
    try:
        return importlib.metadata.version(distribution)
    except importlib.metadata.PackageNotFoundError:
        return None

if not SKIP_INSTALL:
    # Capture every distribution already imported in this runtime, whatever its module name
    # (PIL -> pillow), so a pinned install that replaces a loaded package is detected and the
    # notebook stops with a restart instruction instead of continuing with mixed versions.
    _module_dists = importlib.metadata.packages_distributions()
    _loaded = sorted({d for m in list(sys.modules) for d in _module_dists.get(m.partition('.')[0], ())})
    loaded = {distribution: _installed_version(distribution) for distribution in _loaded}
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', *PINS], check=True)
    importlib.invalidate_caches()
    stale = []
    for distribution, before in loaded.items():
        installed = _installed_version(distribution)
        if before is not None and before != installed:
            stale.append(f'{distribution}: loaded={before}, installed={installed}')
    if stale:
        raise RuntimeError('Core dependencies changed while older modules were loaded: ' + '; '.join(stale) + '. Restart the runtime, then rerun from the top.')

import torch, pandas, sklearn
print({'notebook_source': NOTEBOOK_SOURCE, 'python': platform.python_version(), 'torch': torch.__version__, 'pandas': pandas.__version__, 'sklearn': sklearn.__version__, 'cuda': torch.cuda.is_available()})

## 2. Pipeline code (carried verbatim from `src/tabpfn_regressor_pipeline/` @ `b0f5ba58f9f8`)

The next 1 cell(s) **are** the repository's package, module by module in dependency order: the pinned identity constants, snapshot verification (`verify_snapshot`), staged download (`stage_missing_files`), the named operational ceilings, the public validation and evaluation helpers, and the pipeline class. The text is the modules', byte for byte, except for the rewrite rules listed in `tools/build_notebook.py` (1 rule(s), plus the removal of package-relative `from .x import` lines, whose names are already defined by the preceding cells). The repository's parity test (`tests/test_notebook_parity.py`) fails whenever these cells and the modules diverge, so what you run here is what the repository tests. Nothing in these cells runs a model yet.

**Module 1/1:** `src/tabpfn_regressor_pipeline/pipeline.py`

In [ ]:
"""TabPFN-3 tabular regression — public tutorial API (DIMER pipeline, standalone-notebook carrier).

Inference-only: the pinned TabPFN-3 regressor checkpoint conditions on the labelled training rows **in context**
(``fit`` registers the support set; no gradient update) and predicts query rows in a forward pass. The DIMER
fine-tuning path of this pipeline lives in the private ``tabpfn-regressor-finetuner`` worker and is NOT carried
here (private code; TabPFN-3 weights are non-commercial). This module owns the pinned snapshot scheme, the input
contract (DAT24), the ICL fit / predict / evaluate calls, the ``model.tabpfn_fit`` + ``model.ckpt`` +
``artifact_manifest.json`` bundle the serving path consumes, its pre-load validation, and the evaluation report
(EVAL21). ``tabpfn`` / ``torch`` are imported lazily inside the functions that need them.
"""
# ruff: noqa: E501  -- contract dictionaries and messages are kept on single lines
from __future__ import annotations

import hashlib
import io
import json
import shutil
import tempfile
import zipfile
from collections.abc import Callable, Sequence
from dataclasses import dataclass
from pathlib import Path, PurePosixPath
from typing import Any

import numpy as np
import pandas as pd

MODEL_ID = "Prior-Labs/tabpfn_3"
MODEL_REVISION = "24a16a89d245878b846555110985634aa2e656d7"
# Non-commercial licence: testing, evaluation and internal benchmarking only (no production deployment).
MODEL_LICENSE = "tabpfn-3-license-v1.0"
MODEL_KEY = "tabpfn-3-regressor"
DEFAULT_WEIGHTS_DIR = Path.cwd() / "weights" / MODEL_KEY  # standalone rewrite (build_notebook.py): working-directory-relative
MANIFEST_NAME = "dimer-base-manifest.json"
WEIGHTS_FILE = "tabpfn-v3-regressor-v3_default.ckpt"
WEIGHTS_SHA256 = "311ce18d97e9533d8585eaadafe040fbdd8070533209ed8696641dadc97a7301"
WEIGHTS_BYTES = 233289807
MODEL_VERSION = "v3"
TABPFN_VERSION = "8.1.0"

TASK_TYPE = "tabular_regression"
# Ceilings of the selected generation (contract/model-versions.json, `v3`): rows, features (maxClasses is unused by regression).
MAX_TRAIN_ROWS = 1_000_000
MAX_FEATURES = 2000
MIN_TRAIN_ROWS = 10
DEFAULT_N_ESTIMATORS = 4
DEFAULT_SEED = 42
DEFAULT_VALIDATION_SPLIT = 0.2
MAX_ZIP_EXPANDED_BYTES = 512 * 1024 * 1024  # BYOD ZIP expansion ceiling (512 MiB)
MAX_ZIP_RATIO = 200  # compression-bomb guard: expanded / compressed
DECISION_RULE = "point-estimate"  # predict() returns TabPFN's point estimate (mean of the predictive distribution); no intervals are shipped
METRIC_IDS = ("mae", "rmse", "r2", "mape")
ARTIFACT_SCHEMA_VERSION = 1
FITTED_NAME = "model.tabpfn_fit"
CHECKPOINT_NAME = "model.ckpt"
ARTIFACT_MANIFEST_NAME = "artifact_manifest.json"


# --------------------------------------------------------------------------- snapshot scheme


def sha256_hex(data: bytes) -> str:
    return hashlib.sha256(data).hexdigest()


def sha256_file(path: str | Path) -> str:
    digest = hashlib.sha256()
    with open(path, "rb") as handle:
        for chunk in iter(lambda: handle.read(1 << 20), b""):
            digest.update(chunk)
    return digest.hexdigest()


def _read_manifest(root: Path) -> dict[str, Any]:
    manifest_path = root / MANIFEST_NAME
    if not manifest_path.is_file():
        raise FileNotFoundError(f"snapshot manifest missing: {manifest_path}")
    manifest = json.loads(manifest_path.read_text(encoding="utf-8"))
    if (manifest.get("modelId"), manifest.get("revision")) != (MODEL_ID, MODEL_REVISION):
        raise ValueError(f"manifest names {manifest.get('modelId')}@{manifest.get('revision')}, expected {MODEL_ID}@{MODEL_REVISION}")
    if not isinstance(manifest.get("files"), list) or not manifest["files"]:
        raise ValueError("manifest has no file entries")
    return manifest


def verify_snapshot(weights_dir: str | Path | None = None) -> dict[str, Any]:
    """Re-hash every manifest entry under ``weights_dir``; raise on any missing file, size or digest mismatch."""
    root = Path(weights_dir or DEFAULT_WEIGHTS_DIR)
    manifest = _read_manifest(root)
    checkpoint_ok = False
    for entry in manifest["files"]:
        path = root / entry["path"]
        if not path.is_file():
            raise FileNotFoundError(f"snapshot file missing: {path}")
        size = path.stat().st_size
        if size != entry["bytes"]:
            raise ValueError(f"{entry['path']}: size {size} != manifest {entry['bytes']}")
        digest = sha256_file(path)
        if digest != entry["sha256"]:
            raise ValueError(f"{entry['path']}: sha256 {digest} != manifest {entry['sha256']}")
        if entry["path"] == WEIGHTS_FILE:
            checkpoint_ok = digest == WEIGHTS_SHA256 and size == WEIGHTS_BYTES
    if not checkpoint_ok:
        raise ValueError(f"manifest does not pin {WEIGHTS_FILE} at sha256 {WEIGHTS_SHA256} / {WEIGHTS_BYTES} bytes")
    return manifest


def stage_missing_files(weights_dir: str | Path | None = None, *, allow_download: bool = False, downloader: Callable[..., Any] | None = None) -> list[str]:
    """Fetch the manifest entries absent from ``weights_dir`` (revision-pinned, never ``main``); refuse without ``allow_download``."""
    root = Path(weights_dir or DEFAULT_WEIGHTS_DIR)
    manifest = _read_manifest(root)
    missing = [entry["path"] for entry in manifest["files"] if not (root / entry["path"]).is_file()]
    if not missing:
        return []
    if not allow_download:
        raise FileNotFoundError(f"snapshot at {root} is missing {missing}; pass allow_download=True to stage them from {MODEL_ID}@{MODEL_REVISION[:12]}")
    if downloader is None:
        from huggingface_hub import hf_hub_download

        downloader = hf_hub_download
    for rel in missing:
        downloader(repo_id=MODEL_ID, filename=rel, revision=MODEL_REVISION, local_dir=str(root))
        if not (root / rel).is_file():
            raise FileNotFoundError(f"download did not produce {root / rel}")
    return missing


# --------------------------------------------------------------------------- datasets


def build_synthetic_dataset(out: str | Path, rows: int = 600, seed: int = DEFAULT_SEED) -> Path:
    """Deterministic train/val/test regression ZIP (mirrors examples/build_synthetic_dataset.py); tutorial data only."""
    from sklearn.model_selection import train_test_split

    if rows < 100:
        raise ValueError("rows must be at least 100")
    rng = np.random.default_rng(seed)
    x1 = rng.normal(size=rows)
    x2 = rng.uniform(-2.0, 2.0, size=rows)
    category = rng.choice(["a", "b", "c"], size=rows)
    category_effect = pd.Series(category).map({"a": -2.0, "b": 0.5, "c": 3.0}).to_numpy()
    noise = rng.normal(scale=0.35, size=rows)
    target = 4.0 * x1 - 1.5 * x2 + category_effect + noise - 1.0
    frame = pd.DataFrame({"x1": x1, "x2": x2, "category": category, "target": target})
    train, remainder = train_test_split(frame, test_size=0.3, random_state=seed)
    val, test = train_test_split(remainder, test_size=0.5, random_state=seed)
    out = Path(out).resolve()
    out.parent.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(out, "w", compression=zipfile.ZIP_DEFLATED) as archive:
        for name, part in (("train.csv", train), ("val.csv", val), ("test.csv", test)):
            archive.writestr(name, part.reset_index(drop=True).to_csv(index=False))
    return out


def _safe_member(name: str) -> str:
    path = PurePosixPath(name.replace("\\", "/"))
    if path.is_absolute() or ".." in path.parts or not path.name:
        raise ValueError(f"unsafe archive member: {name!r}")
    return path.name


def safe_extract_zip(zip_path: str | Path, destination: str | Path, max_expanded_bytes: int = MAX_ZIP_EXPANDED_BYTES, max_ratio: int = MAX_ZIP_RATIO) -> list[str]:
    """Extract a ZIP member by member (bare file names only, no directories/absolute/traversing paths); never ``extractall``."""
    zip_path, destination = Path(zip_path), Path(destination)
    destination.mkdir(parents=True, exist_ok=True)
    written: list[str] = []
    with zipfile.ZipFile(zip_path) as archive:
        infos = [info for info in archive.infolist() if not info.is_dir()]
        expanded = sum(info.file_size for info in infos)
        compressed = max(1, sum(info.compress_size for info in infos))
        if expanded > max_expanded_bytes:
            raise ValueError(f"archive expands to {expanded} bytes > ceiling {max_expanded_bytes}")
        if expanded / compressed > max_ratio:
            raise ValueError(f"archive compression ratio {expanded / compressed:.0f} > ceiling {max_ratio}")
        for info in infos:
            name = _safe_member(info.filename)
            with archive.open(info) as source, open(destination / name, "wb") as target:
                shutil.copyfileobj(source, target)
            written.append(name)
    return written


def read_dataset_zip(zip_path: str | Path, *, max_expanded_bytes: int = MAX_ZIP_EXPANDED_BYTES, max_ratio: int = MAX_ZIP_RATIO) -> dict[str, pd.DataFrame]:
    """Read ``train.csv`` (+ optional ``val.csv`` / ``test.csv``) from a ZIP with archive-safety checks; never ``extractall``."""
    zip_path = Path(zip_path)
    frames: dict[str, pd.DataFrame] = {}
    with zipfile.ZipFile(zip_path) as archive:
        infos = [info for info in archive.infolist() if not info.is_dir()]
        expanded = sum(info.file_size for info in infos)
        compressed = max(1, sum(info.compress_size for info in infos))
        if expanded > max_expanded_bytes:
            raise ValueError(f"archive expands to {expanded} bytes > ceiling {max_expanded_bytes}")
        if expanded / compressed > max_ratio:
            raise ValueError(f"archive compression ratio {expanded / compressed:.0f} > ceiling {max_ratio}")
        for info in infos:
            name = _safe_member(info.filename)
            if name in ("train.csv", "val.csv", "test.csv"):
                if name in frames:
                    raise ValueError(f"archive carries {name} more than once")
                frames[name] = pd.read_csv(io.BytesIO(archive.read(info.filename)))
    if "train.csv" not in frames:
        raise ValueError("archive has no train.csv")
    return frames


def read_dataset_dir(dataset_dir: str | Path) -> dict[str, pd.DataFrame]:
    """Read a ZIP (``*.zip``) or loose ``train.csv`` / ``val.csv`` / ``test.csv`` from a directory."""
    root = Path(dataset_dir)
    zips = sorted(root.glob("*.zip"))
    if zips:
        if len(zips) > 1:
            raise ValueError(f"expected one ZIP in {root}, found {len(zips)}")
        return read_dataset_zip(zips[0])
    frames = {name: pd.read_csv(root / name) for name in ("train.csv", "val.csv", "test.csv") if (root / name).is_file()}
    if "train.csv" not in frames:
        raise ValueError(f"{root} has neither a ZIP nor train.csv")
    return frames


def random_holdout(train: pd.DataFrame, target_column: str, validation_split: float = DEFAULT_VALIDATION_SPLIT, seed: int = DEFAULT_SEED) -> tuple[pd.DataFrame, pd.DataFrame]:
    """Seeded random split for a single ``train.csv``; wrong for temporal/grouped data (supply explicit splits instead)."""
    from sklearn.model_selection import train_test_split

    if not 0.0 < validation_split < 1.0:
        raise ValueError("validation_split must be in (0, 1)")
    if target_column not in train.columns:
        raise ValueError(f"target column {target_column!r} not present")
    fit_part, val_part = train_test_split(train, test_size=validation_split, random_state=seed)
    return fit_part.reset_index(drop=True), val_part.reset_index(drop=True)


# --------------------------------------------------------------------------- input contract (DAT24)

INPUT_SCHEMA = {
    "format": "CSV table(s): train.csv with an optional val.csv / test.csv (ZIP or loose files); one row per observation",
    "target": "one numeric target column (declared name), no missing or non-finite values, >= MIN_TRAIN_ROWS training rows, non-constant",
    "features": "every other column; numeric or categorical (strings); unique column names; identical schema across splits",
    "ceilings": {"MAX_TRAIN_ROWS": MAX_TRAIN_ROWS, "MAX_FEATURES": MAX_FEATURES, "MIN_TRAIN_ROWS": MIN_TRAIN_ROWS},
    "splits": "explicit val.csv/test.csv are preserved; without val.csv a seeded random holdout is drawn (independent rows assumed)",
    "reserved_columns": "no `prediction` column in inputs",
}


class InputRejected(ValueError):
    """Raised by ``validate_inputs`` with a structured finding."""

    def __init__(self, finding: dict[str, Any]) -> None:
        super().__init__(finding["message"])
        self.finding = finding


def _reject(code: str, message: str, observed: Any = None) -> InputRejected:
    return InputRejected({"code": code, "verdict": "rejected", "message": message, "observed": observed})


def _check_frame(name: str, frame: pd.DataFrame, target_column: str, feature_columns: list[str] | None) -> list[str]:
    columns = list(frame.columns)
    duplicates = sorted({c for c in columns if columns.count(c) > 1})
    if duplicates:
        raise _reject("DUPLICATE_COLUMNS", f"{name}: duplicate column names", duplicates)
    if target_column not in columns:
        raise _reject("TARGET_MISSING", f"{name}: target column {target_column!r} not present", columns)
    features = [c for c in columns if c != target_column]
    reserved = [c for c in features if c == "prediction"]
    if reserved:
        raise _reject("RESERVED_COLUMNS", f"{name}: reserved output columns present", reserved)
    if feature_columns is not None and features != feature_columns:
        raise _reject("SCHEMA_MISMATCH", f"{name}: feature columns differ from train.csv", {"expected": feature_columns, "observed": features})
    if frame[target_column].isna().any():
        raise _reject("TARGET_MISSING_VALUES", f"{name}: target has missing values", int(frame[target_column].isna().sum()))
    if not pd.api.types.is_numeric_dtype(frame[target_column]):
        raise _reject("TARGET_NOT_NUMERIC", f"{name}: target column {target_column!r} is not numeric", str(frame[target_column].dtype))
    if not np.isfinite(frame[target_column].to_numpy(dtype=float)).all():
        raise _reject("TARGET_NON_FINITE", f"{name}: target has infinite values", target_column)
    if len(frame) == 0:
        raise _reject("EMPTY_SPLIT", f"{name}: no rows", 0)
    return features


def validate_inputs(train: pd.DataFrame, target_column: str, *, val: pd.DataFrame | None = None, test: pd.DataFrame | None = None, names: Sequence[str] | None = None) -> dict[str, Any]:
    """Apply the input contract to the supplied splits and return the DAT24 input manifest (raises ``InputRejected``)."""
    features = _check_frame("train.csv", train, target_column, None)
    if not features:
        raise _reject("NO_FEATURES", "train.csv has no feature columns", list(train.columns))
    if len(features) > MAX_FEATURES:
        raise _reject("TOO_MANY_FEATURES", f"train.csv has {len(features)} features > MAX_FEATURES {MAX_FEATURES}", len(features))
    if len(train) > MAX_TRAIN_ROWS:
        raise _reject("TOO_MANY_ROWS", f"train.csv has {len(train)} rows > MAX_TRAIN_ROWS {MAX_TRAIN_ROWS}", len(train))
    if len(train) < MIN_TRAIN_ROWS:
        raise _reject("TOO_FEW_ROWS", f"train.csv has {len(train)} rows < MIN_TRAIN_ROWS {MIN_TRAIN_ROWS}", len(train))
    target_values = train[target_column].to_numpy(dtype=float)
    if float(np.std(target_values)) == 0.0:
        raise _reject("CONSTANT_TARGET", "train.csv target is constant; nothing to regress", float(target_values[0]))
    findings: list[dict[str, Any]] = []
    target_stats = {"mean": float(np.mean(target_values)), "std": float(np.std(target_values)), "min": float(np.min(target_values)), "max": float(np.max(target_values))}
    splits: dict[str, Any] = {"train": {"rows": int(len(train)), "target": target_stats}}
    for name, frame in (("val.csv", val), ("test.csv", test)):
        if frame is None:
            continue
        _check_frame(name, frame, target_column, features)
        values = frame[target_column].to_numpy(dtype=float)
        if values.min() < target_stats["min"] or values.max() > target_stats["max"]:
            findings.append({"input": name, "verdict": "warning", "code": "TARGET_OUT_OF_TRAINING_RANGE", "message": f"{name} targets fall outside the training range; extrapolation is being scored", "observed": {"min": float(values.min()), "max": float(values.max())}})
        splits[name.split(".")[0]] = {"rows": int(len(frame)), "target": {"mean": float(np.mean(values)), "std": float(np.std(values)), "min": float(values.min()), "max": float(values.max())}}
    numeric = [c for c in features if pd.api.types.is_numeric_dtype(train[c])]
    for c in numeric:
        values = train[c].dropna().to_numpy(dtype=float)
        if values.size and not np.isfinite(values).all():
            raise _reject("NON_FINITE_FEATURE", f"train.csv numeric feature {c!r} contains infinite values", c)
    if (target_values == 0).mean() > 0.5:
        findings.append({"input": "train.csv", "verdict": "warning", "code": "ZERO_HEAVY_TARGET", "message": "more than half of the training targets are zero; mape is computed over non-zero rows only", "observed": float((target_values == 0).mean())})
    return {
        "schema": dict(INPUT_SCHEMA),
        "inputs": [{"id": names[0] if names else "dataset-0", "mode": "in-context-fit", "target_column": target_column, "feature_columns": features, "numeric_features": len(numeric), "categorical_features": len(features) - len(numeric), "target_stats": target_stats, "splits": splits, "train_sha256": sha256_hex(train.to_csv(index=False).encode("utf-8"))}],
        "verdict": "accepted",
        "findings": findings,
        "model_id": MODEL_ID,
        "model_revision": MODEL_REVISION,
        "model_version": MODEL_VERSION,
    }


def validate_new_rows(frame: pd.DataFrame, feature_columns: Sequence[str], *, target_column: str | None = None) -> pd.DataFrame:
    """Rows for inference must carry exactly the artifact's feature columns and no target/output columns."""
    columns = list(frame.columns)
    duplicates = sorted({c for c in columns if columns.count(c) > 1})
    if duplicates:
        raise _reject("DUPLICATE_COLUMNS", "new rows: duplicate column names", duplicates)
    reserved = [c for c in [target_column, "prediction"] if c and c in columns]
    if reserved:
        raise _reject("RESERVED_COLUMNS", "new rows: remove target/prediction columns before inference", reserved)
    missing = [c for c in feature_columns if c not in columns]
    extra = [c for c in columns if c not in feature_columns]
    if missing or extra:
        raise _reject("SCHEMA_MISMATCH", "new rows: feature schema mismatch", {"missing": missing, "extra": extra})
    if len(frame) == 0:
        raise _reject("EMPTY_INPUT", "new rows: no rows", 0)
    frame = frame.loc[:, list(feature_columns)].copy()
    for c in frame.select_dtypes(include=np.number).columns:
        values = frame[c].dropna().to_numpy(dtype=float)
        if values.size and not np.isfinite(values).all():
            raise _reject("NON_FINITE_FEATURE", f"new rows: numeric feature {c!r} contains infinite values", c)
    return frame


# --------------------------------------------------------------------------- metrics / baseline


def regression_metrics(y_true: Sequence[Any], y_pred: Sequence[Any]) -> dict[str, float]:
    """MAE, RMSE, R² and MAPE (the latter over non-zero targets only; absent when every target is zero)."""
    from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

    yt = np.asarray(y_true, dtype=float)
    yp = np.asarray(y_pred, dtype=float)
    if yt.shape != yp.shape or yt.size == 0:
        raise ValueError("y_true and y_pred must be equal-length, non-empty 1-D arrays")
    out = {"mae": float(mean_absolute_error(yt, yp)), "rmse": float(np.sqrt(mean_squared_error(yt, yp))), "r2": float(r2_score(yt, yp)) if yt.size > 1 and float(np.std(yt)) > 0 else float("nan")}
    nonzero = yt != 0
    if nonzero.any():
        out["mape"] = float(np.mean(np.abs((yt[nonzero] - yp[nonzero]) / yt[nonzero])))
    return out


def mean_baseline(train_targets: Sequence[Any], eval_targets: Sequence[Any]) -> dict[str, Any]:
    """Always predict the training mean, scored on the evaluation rows."""
    train_targets = np.asarray(train_targets, dtype=float)
    eval_targets = np.asarray(eval_targets, dtype=float)
    if train_targets.size == 0 or eval_targets.size == 0:
        raise ValueError("baseline needs training and evaluation targets")
    mean = float(train_targets.mean())
    metrics = regression_metrics(eval_targets, np.full(eval_targets.shape, mean))
    return {"trainMean": mean, **metrics}


# --------------------------------------------------------------------------- artifact bundle


def _safe_fitted_archive(path: Path) -> dict[str, Any]:
    with zipfile.ZipFile(path) as archive:
        names = archive.namelist()
        for name in names:
            _safe_member(name)
        if "init_params.json" not in names:
            raise ValueError("fitted archive lacks init_params.json")
        return json.loads(archive.read("init_params.json"))


def manifest_digest(manifest: dict[str, Any], key: str) -> str:
    """Both manifest shapes: flat ``<key>Sha256`` (this module, the worker) or nested ``sha256[<key>]``."""
    value = manifest.get(key + "Sha256") or (manifest.get("sha256") or {}).get(key)
    if not isinstance(value, str) or len(value) != 64:
        raise ValueError(f"manifest has no SHA-256 for {key}")
    return value


def validate_artifact_bundle(artifact_dir: str | Path, *, expected_fitted_sha256: str = "", expected_checkpoint_sha256: str = "") -> dict[str, Any]:
    """Check manifest schema, member names, sizes, digests and archive safety BEFORE any model state is deserialised."""
    root = Path(artifact_dir)
    manifest_path = root / ARTIFACT_MANIFEST_NAME
    if not manifest_path.is_file():
        raise FileNotFoundError(f"{ARTIFACT_MANIFEST_NAME} missing in {root}")
    manifest = json.loads(manifest_path.read_text(encoding="utf-8"))
    if manifest.get("schemaVersion") != ARTIFACT_SCHEMA_VERSION or manifest.get("taskType") != TASK_TYPE:
        raise ValueError(f"unsupported manifest: schemaVersion={manifest.get('schemaVersion')} taskType={manifest.get('taskType')}")
    for key in ("targetColumn", "featureColumns"):
        if key not in manifest:
            raise ValueError(f"manifest lacks {key}")
    digests: dict[str, str] = {}
    for key, override in (("fittedEstimator", expected_fitted_sha256), ("foundationCheckpoint", expected_checkpoint_sha256)):
        name = manifest.get(key)
        if not isinstance(name, str) or Path(name).name != name:
            raise ValueError(f"manifest {key} must be a bare file name, got {name!r}")
        path = root / name
        if not path.is_file() or path.stat().st_size < 1024:
            raise ValueError(f"{name} is missing or implausibly small")
        digest = sha256_file(path)
        if digest != manifest_digest(manifest, key):
            raise ValueError(f"{name} SHA-256 {digest} != manifest {manifest_digest(manifest, key)}")
        if override and digest != override.strip().lower():
            raise ValueError(f"{name} SHA-256 {digest} != expected {override.strip().lower()}")
        digests[key] = digest
    init_params = _safe_fitted_archive(root / manifest["fittedEstimator"])
    return {**manifest, "verifiedSha256": digests, "recordedModelPath": init_params.get("model_path")}


def rewrite_model_path(fitted_archive: Path, checkpoint: Path, destination: Path) -> Path:
    """Copy a ``.tabpfn_fit`` archive while pointing its ``init_params.json`` ``model_path`` at ``checkpoint`` (original untouched)."""
    fitted_archive, checkpoint, destination = fitted_archive.resolve(), checkpoint.resolve(), destination.resolve()
    if not fitted_archive.is_file():
        raise FileNotFoundError(f"fitted estimator not found: {fitted_archive}")
    if not checkpoint.is_file():
        raise FileNotFoundError(f"companion checkpoint not found: {checkpoint}")
    saw_init = False
    destination.parent.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(fitted_archive, "r") as source, zipfile.ZipFile(destination, "w", compression=zipfile.ZIP_DEFLATED) as target:
        for info in source.infolist():
            _safe_member(info.filename)
            payload = source.read(info.filename)
            if info.filename == "init_params.json":
                params = json.loads(payload.decode("utf-8"))
                params["model_path"] = str(checkpoint)
                payload = json.dumps(params, sort_keys=True).encode("utf-8")
                saw_init = True
            target.writestr(info, payload)
    if not saw_init:
        destination.unlink(missing_ok=True)
        raise ValueError("fitted artifact does not contain init_params.json")
    return destination


def zip_artifact_bundle(artifact_dir: str | Path, zip_path: str | Path) -> str:
    """Zip the three bundle members flat (bare names) for transport to the artifact-inference notebook; returns the ZIP SHA-256."""
    root, zip_path = Path(artifact_dir), Path(zip_path)
    zip_path.parent.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(zip_path, "w", compression=zipfile.ZIP_DEFLATED) as archive:
        for name in (ARTIFACT_MANIFEST_NAME, FITTED_NAME, CHECKPOINT_NAME):
            archive.write(root / name, arcname=name)
    return sha256_file(zip_path)


def _coerce_non_json_init_params(model: Any) -> None:
    """tabpfn 8.1.0 serialises ``get_params()`` as JSON; stringify the values that are not JSON-encodable (e.g. a Path)."""
    for param, value in model.get_params(deep=False).items():
        try:
            json.dumps(value)
        except (TypeError, ValueError):
            setattr(model, param, str(value))


# --------------------------------------------------------------------------- pipeline


@dataclass
class TabPFNRegressorPipeline:
    """The verified TabPFN-3 checkpoint plus, after ``fit`` or ``from_artifact``, an in-context-fitted estimator."""

    weights_path: Path
    device: str = "cpu"
    source: str = "local-snapshot"
    n_estimators: int = DEFAULT_N_ESTIMATORS
    random_state: int = DEFAULT_SEED
    target_column: str | None = None
    feature_columns: list[str] | None = None
    target_stats: dict[str, float] | None = None
    _model: Any = None

    @classmethod
    def from_pretrained(cls, device: str | None = None, weights_dir: str | Path | None = None, allow_download: bool = False, *, n_estimators: int = DEFAULT_N_ESTIMATORS, random_state: int = DEFAULT_SEED) -> TabPFNRegressorPipeline:
        root = Path(weights_dir or DEFAULT_WEIGHTS_DIR)
        stage_missing_files(root, allow_download=allow_download)
        verify_snapshot(root)
        if device is None:
            import torch

            device = "cuda" if torch.cuda.is_available() else "cpu"
        return cls(weights_path=root / WEIGHTS_FILE, device=device, n_estimators=n_estimators, random_state=random_state)

    def _build_estimator(self) -> Any:
        from tabpfn import TabPFNRegressor

        return TabPFNRegressor(model_path=str(self.weights_path), device=self.device, n_estimators=self.n_estimators, random_state=self.random_state, show_progress_bar=False)

    def fit(self, X: pd.DataFrame, y: Sequence[Any], *, target_column: str = "target", estimator_factory: Callable[[], Any] | None = None) -> TabPFNRegressorPipeline:
        """In-context fit: register the training rows as the support set. No gradient update is performed."""
        model = (estimator_factory or self._build_estimator)()
        targets = pd.Series(np.asarray(list(y), dtype=float), name=target_column)
        model.fit(X, targets)
        self._model = model
        self.target_column = target_column
        self.feature_columns = list(X.columns)
        values = targets.to_numpy()
        self.target_stats = {"mean": float(values.mean()), "std": float(values.std()), "min": float(values.min()), "max": float(values.max())}
        self.source = "in-context-fit"
        return self

    def _require(self) -> Any:
        if self._model is None or self.feature_columns is None:
            raise RuntimeError("no fitted estimator: call fit() or from_artifact() first")
        return self._model

    def predict_values(self, X: pd.DataFrame) -> np.ndarray:
        model = self._require()
        X = validate_new_rows(X, self.feature_columns or [], target_column=self.target_column)
        return np.asarray(model.predict(X), dtype=float).reshape(-1)

    def predict(self, X: pd.DataFrame) -> pd.DataFrame:
        """``prediction`` = TabPFN's point estimate per row (no intervals are shipped)."""
        values = self.predict_values(X)
        out = pd.DataFrame({"row_id": np.asarray(X.index), "prediction": values})
        out.attrs["decision_rule"] = DECISION_RULE
        return out

    def evaluate(self, X: pd.DataFrame, y: Sequence[Any]) -> dict[str, float]:
        return regression_metrics(np.asarray(list(y), dtype=float), self.predict_values(X))

    def save_artifact(self, artifact_dir: str | Path, *, saver: Callable[[Any, Path], None] | None = None) -> dict[str, Any]:
        """Write ``model.tabpfn_fit`` + ``model.ckpt`` (byte copy of the verified checkpoint) + ``artifact_manifest.json``."""
        model = self._require()
        root = Path(artifact_dir)
        root.mkdir(parents=True, exist_ok=True)
        if saver is None:
            from tabpfn.model_loading import save_fitted_tabpfn_model

            def saver(estimator: Any, path: Path) -> None:
                _coerce_non_json_init_params(estimator)
                save_fitted_tabpfn_model(estimator, path)

        saver(model, root / FITTED_NAME)
        shutil.copyfile(self.weights_path, root / CHECKPOINT_NAME)
        manifest = {
            "schemaVersion": ARTIFACT_SCHEMA_VERSION,
            "taskType": TASK_TYPE,
            "targetColumn": self.target_column,
            "featureColumns": list(self.feature_columns or []),
            "targetStats": dict(self.target_stats or {}),
            "fittedEstimator": FITTED_NAME,
            "foundationCheckpoint": CHECKPOINT_NAME,
            "fittedEstimatorSha256": sha256_file(root / FITTED_NAME),
            "foundationCheckpointSha256": sha256_file(root / CHECKPOINT_NAME),
            "portableLoader": "tabpfn_regressor_pipeline.TabPFNRegressorPipeline.from_artifact",
            "baseModel": {"modelId": MODEL_ID, "revision": MODEL_REVISION, "file": WEIGHTS_FILE, "sha256": WEIGHTS_SHA256, "modelVersion": MODEL_VERSION, "license": MODEL_LICENSE},
            "mode": "zero-shot-icl",
            "nEstimators": self.n_estimators,
            "randomState": self.random_state,
        }
        (root / ARTIFACT_MANIFEST_NAME).write_text(json.dumps(manifest, indent=2) + "\n", encoding="utf-8")
        return manifest

    @classmethod
    def from_artifact(cls, artifact_dir: str | Path, device: str | None = None, *, expected_fitted_sha256: str = "", expected_checkpoint_sha256: str = "", loader: Callable[[Path, str], Any] | None = None) -> TabPFNRegressorPipeline:
        """Validate the bundle, then reconstruct the estimator from the fitted archive + companion checkpoint (no refit, no download)."""
        root = Path(artifact_dir)
        manifest = validate_artifact_bundle(root, expected_fitted_sha256=expected_fitted_sha256, expected_checkpoint_sha256=expected_checkpoint_sha256)
        if device is None:
            import torch

            device = "cuda" if torch.cuda.is_available() else "cpu"
        if loader is None:
            from tabpfn.model_loading import load_fitted_tabpfn_model

            def loader(fitted: Path, dev: str) -> Any:
                return load_fitted_tabpfn_model(fitted, device=dev)

        checkpoint = root / manifest["foundationCheckpoint"]
        with tempfile.TemporaryDirectory() as temp_dir:
            rewritten = rewrite_model_path(root / manifest["fittedEstimator"], checkpoint, Path(temp_dir) / FITTED_NAME)
            model = loader(rewritten, device)
        if not hasattr(model, "predict"):
            raise RuntimeError("reconstructed estimator has no predict method")
        return cls(weights_path=checkpoint, device=device, source="artifact", n_estimators=int(manifest.get("nEstimators", DEFAULT_N_ESTIMATORS)), random_state=int(manifest.get("randomState", DEFAULT_SEED)), target_column=manifest["targetColumn"], feature_columns=list(manifest["featureColumns"]), target_stats=dict(manifest.get("targetStats") or {}), _model=model)


# --------------------------------------------------------------------------- evaluation report (EVAL21)


def evaluation_report(metrics: dict[str, float] | None, *, baseline: dict[str, Any] | None = None, n_validation: int = 0, target_column: str | None = None, sample_kind: str = "synthetic", reload_check: dict[str, Any] | None = None, split_name: str = "val.csv") -> dict[str, Any]:
    """Evaluation stage: the metrics are honest about what they are (single holdout, tutorial sample) or ``not-measurable``."""
    base = {
        "task": TASK_TYPE,
        "model_id": MODEL_ID,
        "model_revision": MODEL_REVISION,
        "model_version": MODEL_VERSION,
        "decision_rule": DECISION_RULE,
        "adaptation": "in-context conditioning only (no gradient update); the private-worker fine-tune path is not carried",
        "sample_kind": sample_kind,
        "n_validation": int(n_validation),
        "split": split_name,
        "target_column": target_column,
        "reload_check": reload_check,
        "caveats": ["single holdout, no dispersion estimate", "point estimates only; no prediction intervals are shipped", "mape is computed over non-zero targets only", "synthetic sample metrics are plumbing evidence only" if sample_kind == "synthetic" else "BYOD metrics are one holdout of one table"],
    }
    if not metrics or n_validation == 0:
        return {**base, "metrics": [], "verdict": "not-measurable", "reason": "no labelled validation split was scored", "needs": "a labelled, leakage-safe holdout from the deployment domain scored with mae / rmse / r2 against mean_baseline; repeated splits for any dispersion estimate"}
    entries = [{"id": k, "value": float(v)} for k, v in metrics.items() if k in METRIC_IDS]
    return {**base, "metrics": entries, "baselines": {"training_mean": baseline} if baseline else {}, "verdict": "sample-sanity", "reason": f"{n_validation} validation row(s) from one holdout; tutorial evidence, not a benchmark", "needs": "a domain-representative labelled test set, subgroup breakdowns, residual analysis on held-out data and repeated splits for any generalisable claim"}

## 3. Pin, stage and verify the model

The model identity is carried twice — `MODEL_ID`/`MODEL_REVISION` in the module above and the `4`-file manifest below (paths, byte sizes, SHA-256) — and the cell first asserts they agree. It writes the manifest into the working-directory snapshot, then `stage_missing_files(..., allow_download=True)` fetches exactly the entries that are absent from the Hugging Face Hub **at revision `24a16a89d245…`** (never `main`), `verify_snapshot` re-hashes every file and raises on the first size or digest mismatch, and only then does `TabPFNRegressorPipeline.from_pretrained(weights_dir=WEIGHTS_DIR)` load the verified files. There is no fallback to a different download and no remote model code is executed. The effective identity, device and weight source are printed before any inference.

In [ ]:
import json

MANIFEST = {
  "format": "dimer_hf_snapshot",
  "formatVersion": 1,
  "modelKey": "tabpfn-3-regressor",
  "modelId": "Prior-Labs/tabpfn_3",
  "revision": "24a16a89d245878b846555110985634aa2e656d7",
  "files": [
    {
      "path": "LICENSE",
      "bytes": 16794,
      "sha256": "dca491280b68f471312a15d54add7b8e724adf19fb0e113544b1ef91e060f5d5"
    },
    {
      "path": "README.md",
      "bytes": 8377,
      "sha256": "53d61ea9bbf605c42892d13c53afb78e5ba8a6d1513d89db86736a620d3c9e81"
    },
    {
      "path": "config.json",
      "bytes": 33,
      "sha256": "d9bc48f72a18bcbbb0a58dbe1ca7ac4123b9cfa1b7c0a79da5bcf3543cb32344"
    },
    {
      "path": "tabpfn-v3-regressor-v3_default.ckpt",
      "bytes": 233289807,
      "sha256": "311ce18d97e9533d8585eaadafe040fbdd8070533209ed8696641dadc97a7301"
    }
  ],
  "totalBytes": 233315011
}

if (MANIFEST['modelId'], MANIFEST['revision']) != (MODEL_ID, MODEL_REVISION):
    raise RuntimeError('inline manifest does not name the identity carried by the pipeline module; the notebook was not regenerated after a change')
WEIGHTS_DIR = DEFAULT_WEIGHTS_DIR
WEIGHTS_DIR.mkdir(parents=True, exist_ok=True)
with open(WEIGHTS_DIR / MANIFEST_NAME, 'w', encoding='utf-8') as handle:
    json.dump(MANIFEST, handle, indent=2)
print({'model_id': MODEL_ID, 'revision': MODEL_REVISION, 'license': MODEL_LICENSE, 'files': len(MANIFEST['files']), 'total_bytes': MANIFEST['totalBytes']})
fetched = stage_missing_files(WEIGHTS_DIR, allow_download=True)
print({'weights_dir': str(WEIGHTS_DIR), 'fetched': fetched})
snapshot = verify_snapshot(WEIGHTS_DIR)
_files = snapshot.get('files', []) if isinstance(snapshot, dict) else []
print({'verified_files': len(_files) if isinstance(_files, list) else _files, 'revision': snapshot.get('revision', MODEL_REVISION) if isinstance(snapshot, dict) else MODEL_REVISION})
pipe = TabPFNRegressorPipeline.from_pretrained(weights_dir=WEIGHTS_DIR)
print({'device': getattr(pipe, 'device', None), 'source': getattr(pipe, 'source', 'local-snapshot')})

## 4. Supply the external bundle and validate it before any model state is reconstructed

The bundle ZIP comes from `ARTIFACT_ZIP_PATH` (an executor places it there) or from the upload dialog; an optional `EXPECTED_ZIP_SHA256` and the two member digests `EXPECTED_FITTED_SHA256` / `EXPECTED_CKPT_SHA256` — pasted from the producer's record — fail closed on mismatch. `safe_extract_zip` extracts member by member (bare file names only, expanded-size and ratio ceilings; never `extractall`), then `validate_artifact_bundle` checks the manifest schema and task type, that both binary members are named by the manifest, exist and are plausibly sized, that their SHA-256 digests match the manifest (and the expected values), and that the fitted archive is a ZIP whose members are all safe relative paths and which carries `init_params.json` — all **before** anything is deserialised. The bundled `model.ckpt` must equal the pinned checkpoint of Section 3 (`WEIGHTS_SHA256`): a bundle produced on other weights is refused. Look for the feature schema and the training target range the artifact records.

In [ ]:
ARTIFACT_ZIP_PATH = ''  # @param {type:"string"}
EXPECTED_ZIP_SHA256 = ''  # @param {type:"string"}
EXPECTED_FITTED_SHA256 = ''  # @param {type:"string"}
EXPECTED_CKPT_SHA256 = ''  # @param {type:"string"}

os.makedirs('outputs', exist_ok=True)
WORK = Path('work')
shutil.rmtree(WORK, ignore_errors=True)
WORK.mkdir(parents=True)
if ARTIFACT_ZIP_PATH:
    zip_name, zip_payload = Path(ARTIFACT_ZIP_PATH).name, Path(ARTIFACT_ZIP_PATH).read_bytes()
    artifact_source = f'path: {ARTIFACT_ZIP_PATH}'
else:
    from google.colab import files
    uploaded = files.upload()
    if len(uploaded) != 1:
        raise RuntimeError('Upload exactly one bundle ZIP.')
    zip_name, zip_payload = next(iter(uploaded.items()))
    artifact_source = 'upload dialog'
zip_path = WORK / Path(zip_name).name
zip_path.write_bytes(zip_payload)
zip_sha256 = sha256_file(zip_path)
for label, expected in (('EXPECTED_ZIP_SHA256', EXPECTED_ZIP_SHA256), ('EXPECTED_FITTED_SHA256', EXPECTED_FITTED_SHA256), ('EXPECTED_CKPT_SHA256', EXPECTED_CKPT_SHA256)):
    expected = expected.strip().lower()
    if expected and (len(expected) != 64 or any(character not in '0123456789abcdef' for character in expected)):
        raise ValueError(f'{label} must be 64 hex chars')
if EXPECTED_ZIP_SHA256 and zip_sha256 != EXPECTED_ZIP_SHA256.strip().lower():
    raise RuntimeError('Artifact ZIP SHA-256 mismatch')
ARTIFACT_DIR = WORK / 'external-artifact'
members = safe_extract_zip(zip_path, ARTIFACT_DIR)
required = {ARTIFACT_MANIFEST_NAME, FITTED_NAME, CHECKPOINT_NAME}
if not required <= set(members):
    raise ValueError(f'bundle must carry {sorted(required)}; got {sorted(members)}')
artifact = validate_artifact_bundle(ARTIFACT_DIR, expected_fitted_sha256=EXPECTED_FITTED_SHA256, expected_checkpoint_sha256=EXPECTED_CKPT_SHA256 or WEIGHTS_SHA256)
if artifact['verifiedSha256']['foundationCheckpoint'] != WEIGHTS_SHA256:
    raise RuntimeError('the bundled model.ckpt is not the pinned TabPFN-3 checkpoint carried by this notebook')
FEATURE_COLUMNS = list(artifact['featureColumns'])
TARGET_COLUMN = artifact['targetColumn']
TARGET_STATS = dict(artifact.get('targetStats') or {})
print({'artifact_source': artifact_source, 'zip': zip_name, 'zip_sha256': zip_sha256[:16], 'members': sorted(members), 'mode': artifact.get('mode'), 'portableLoader': artifact.get('portableLoader'), 'baseModel': artifact.get('baseModel')})
print({'featureColumns': FEATURE_COLUMNS, 'targetColumn': TARGET_COLUMN, 'targetStats': TARGET_STATS, 'fittedSha256': artifact['verifiedSha256']['fittedEstimator'][:16], 'ckptSha256': artifact['verifiedSha256']['foundationCheckpoint'][:16], 'recordedModelPath': artifact['recordedModelPath']})

## 5. Reconstruct the serving estimator from the bundle alone

`TabPFNRegressorPipeline.from_artifact` re-runs the bundle validation, rewrites the fitted archive's recorded `model_path` — in a temporary copy, never the original — to point at the companion `model.ckpt` beside it, and calls TabPFN's `load_fitted_tabpfn_model`. No network download is attempted for the foundation weights: they come from the bundle (and were proven equal to the pinned checkpoint). The in-context training rows travel inside the fitted archive; nothing is refit here. The reconstructed feature schema must agree with the manifest, else the notebook stops.

In [ ]:
fresh = TabPFNRegressorPipeline.from_artifact(ARTIFACT_DIR, device=pipe.device, expected_checkpoint_sha256=WEIGHTS_SHA256)
if fresh.feature_columns != FEATURE_COLUMNS or fresh.target_column != TARGET_COLUMN:
    raise RuntimeError('reconstructed estimator disagrees with the manifest')
print({'source': fresh.source, 'device': fresh.device, 'target_stats': fresh.target_stats, 'n_estimators': fresh.n_estimators, 'random_state': fresh.random_state, 'refit': False, 'network_fallback_for_weights': False})

## 6. Supply new unlabelled rows → validate → input manifest

Upload one CSV (or point `NEW_DATA_PATH` at one) containing exactly the artifact's feature columns and no target or `prediction` column. `validate_new_rows` rejects duplicate, missing or extra columns and infinite numeric values rather than silently dropping anything; the resulting input manifest names the schema the artifact imposes, the row count, the column types and the verdict, and is written to `outputs/tabpfn_regressor_artifact_inference_input_manifest.json`. To show what rejection looks like, the cell also validates a probe with an extra column and records the structured finding.

In [ ]:
NEW_DATA_PATH = ''  # @param {type:"string"}

if NEW_DATA_PATH:
    new_name, new_payload = Path(NEW_DATA_PATH).name, Path(NEW_DATA_PATH).read_bytes()
else:
    from google.colab import files
    uploaded = files.upload()
    if len(uploaded) != 1:
        raise RuntimeError('Upload exactly one CSV of new rows.')
    new_name, new_payload = next(iter(uploaded.items()))
raw_rows = pd.read_csv(io.BytesIO(new_payload))
new_rows = validate_new_rows(raw_rows, FEATURE_COLUMNS, target_column=TARGET_COLUMN)
input_manifest = {
    'schema': {'format': 'CSV of unlabelled rows', 'columns': 'exactly the artifact feature columns, in any order', 'reserved': [TARGET_COLUMN, 'prediction'], 'numeric': 'finite values'},
    'inputs': [{'id': new_name, 'mode': 'artifact-inference', 'rows': int(len(new_rows)), 'feature_columns': FEATURE_COLUMNS, 'numeric_features': int(len(new_rows.select_dtypes(include=np.number).columns)), 'missing_values': {k: int(v) for k, v in new_rows.isna().sum().items() if v}, 'sha256': sha256_hex(new_payload)}],
    'verdict': 'accepted',
    'findings': [],
    'model_id': MODEL_ID,
    'model_revision': MODEL_REVISION,
}
try:
    validate_new_rows(new_rows.assign(unexpected_column=0), FEATURE_COLUMNS, target_column=TARGET_COLUMN)
except InputRejected as exc:
    input_manifest['findings'].append({'input': 'extra-column-probe', **exc.finding})
with open('outputs/tabpfn_regressor_artifact_inference_input_manifest.json', 'w', encoding='utf-8') as handle:
    json.dump(input_manifest, handle, indent=2, ensure_ascii=False, default=str)
print(json.dumps(input_manifest['inputs'][0], indent=2, default=str))
print('findings:', json.dumps(input_manifest['findings'], indent=2, default=str))

## 7. Predict, report what cannot be measured, and export

`predict` returns `prediction`, TabPFN's point estimate in target units; **no prediction interval** or calibrated uncertainty is shipped, and values outside the artifact's recorded training target range are extrapolations. Because the rows carry no labels, `evaluation_report` records the verdict `not-measurable` and states what labelled data would make the task measurable — it does not invent a score. It is written to `outputs/tabpfn_regressor_artifact_inference_evaluation_report.json`; the predictions go to `outputs/tabpfn_regressor_artifact_inference_predictions.csv`, and `outputs/tabpfn_regressor_artifact_inference_result.json` records the artifact identity and verified digests, the input manifest, the notebook's source, the pinned model identity, revision and licence, and the runtime. No credentials are recorded.

In [ ]:
predictions = fresh.predict(new_rows)
predictions.to_csv('outputs/tabpfn_regressor_artifact_inference_predictions.csv', index=False)
report = evaluation_report(None, n_validation=0, target_column=TARGET_COLUMN, sample_kind='BYOD')
with open('outputs/tabpfn_regressor_artifact_inference_evaluation_report.json', 'w', encoding='utf-8') as handle:
    json.dump(report, handle, indent=2, ensure_ascii=False)
payload = {
    'artifact': {k: artifact.get(k) for k in ('schemaVersion', 'taskType', 'targetColumn', 'featureColumns', 'targetStats', 'fittedEstimator', 'foundationCheckpoint', 'portableLoader', 'baseModel', 'mode', 'nEstimators', 'randomState')},
    'artifact_verified_sha256': artifact['verifiedSha256'],
    'artifact_zip': {'name': zip_name, 'sha256': zip_sha256, 'source': artifact_source},
    'reconstruction': {'loader': 'tabpfn_regressor_pipeline.TabPFNRegressorPipeline.from_artifact', 'device': fresh.device, 'network_fallback_for_weights': False, 'refit': False, 'decision_rule': DECISION_RULE},
    'input_manifest': input_manifest,
    'evaluation_report': report,
    'scored_rows': int(len(predictions)),
    'notebook_source': NOTEBOOK_SOURCE,
    'repository_revision': NOTEBOOK_SOURCE['repository_revision'],
    'model_id': MODEL_ID,
    'model_revision': MODEL_REVISION,
    'model_license': MODEL_LICENSE,
    'model_file': WEIGHTS_FILE,
    'runtime': {'python': platform.python_version(), 'torch': torch.__version__, 'tabpfn': importlib.metadata.version('tabpfn'), 'pandas': pd.__version__, 'device': fresh.device},
}
with open('outputs/tabpfn_regressor_artifact_inference_result.json', 'w', encoding='utf-8') as handle:
    json.dump(payload, handle, indent=2, ensure_ascii=False, default=str)
print(predictions.head(8).to_string(index=False))
print({'verdict': report['verdict'], 'needs': report['needs']})
print(sorted(os.listdir('outputs')))

## Interpretation and limits

A successful run proves that an independently supplied bundle is internally consistent with its manifest and was produced on the pinned TabPFN-3 checkpoint, that the carried module reconstructs the estimator from the bundle alone without refitting or downloading weights, and that schema-compatible new rows can be scored and exported. It does **not** authenticate the producer, make untrusted serialised estimator state safe to load, or establish predictive quality, calibration, fairness or production fitness — the evaluation report is `not-measurable` by construction. Never bypass a failed digest, schema or archive-safety check; obtain a correct bundle from a trusted producer.

Successful execution proves that the recorded repository revision's pipeline module, carried in this notebook, can acquire and digest-verify the pinned checkpoint, validate an external bundle before deserialisation, reconstruct the serving estimator, validate new rows and emit the shown machine-readable outputs in the tested runtime — without the repository being reachable. It does **not** establish benchmark superiority or anything about the quality of the artifact's training rows.

**Next experiments:** paste the producer's digests into the expected-digest fields and watch a tampered bundle fail closed; supply rows with a missing feature column and read the structured rejection; compare the point estimates of the same rows produced by the task-inference notebook's in-memory estimator and by this reconstruction (they should agree to floating-point precision).

## References

- Repository README: https://github.com/kurtvalcorza/tabpfn-regressor-pipeline/blob/main/README.md
- Repository model card: https://github.com/kurtvalcorza/tabpfn-regressor-pipeline/blob/main/MODEL_CARD.md
- Weight provenance: https://github.com/kurtvalcorza/tabpfn-regressor-pipeline/blob/main/docs/WEIGHTS.md
- Upstream model: https://huggingface.co/Prior-Labs/tabpfn_3
- Upstream code: https://github.com/PriorLabs/TabPFN
- TabPFN-3 technical report: https://arxiv.org/abs/2605.13986